# Connection
Opens one connection to the SQL warehouse used for every notebook in this folder, from a local Jupyter kernel. Values come from the environment; only the variable names are here: `DATABRICKS_SERVER_HOSTNAME`, `DATABRICKS_HTTP_PATH`, `DATABRICKS_TOKEN` (unset → browser OAuth). The principal should hold only `USE CATALOG`, `USE SCHEMA` and `SELECT` on `system` and `CAN USE` on the warehouse. Needs `databricks-sql-connector[pyarrow]` (connector 4.x ships pyarrow as an extra).

Question notebooks run by hand can start with `%run ./00_connection`; notebooks run by `run_cells.py` repeat the setup cell instead. Inside a Databricks workspace this notebook only keeps the visibility checks: the cluster or warehouse is already attached, so queries run as `spark.sql(q, args={...})` or `%sql` cells.

In [ ]:
import os
import secrets
from databricks import sql  # pip install "databricks-sql-connector[pyarrow]" (4.x)

TOKEN = os.environ.get("DATABRICKS_TOKEN")
AUTH = {"access_token": TOKEN} if TOKEN else {"auth_type": "databricks-oauth"}
conn = sql.connect(server_hostname=os.environ["DATABRICKS_SERVER_HOSTNAME"],
                   http_path=os.environ["DATABRICKS_HTTP_PATH"], **AUTH)

def q(statement, **params):
    """Run one read-only statement with named parameters (:name) and return a DataFrame."""
    with conn.cursor() as cur:
        cur.execute(statement, params or None)
        return cur.fetchall_arrow().to_pandas()

# Keyed-hash key for masking identities (bind as :k, never print it); new per kernel.
PII_KEY = secrets.token_hex(32)

**Does:** confirms who is connected and on which catalog.
**Reads:** `current_user()`, `current_catalog()` · one row.
**Look for:** the read-only service principal or your own user, never an admin principal.

In [ ]:
q(r"""
SELECT current_user() AS principal, current_catalog() AS catalog
""")

**Does:** lists the system tables this principal can see.
**Reads:** `system.information_schema.tables` · one row per visible table · privilege-aware, live.
**Look for:** every table the questions need. A missing one is "not visible" (schema off, no grant or another region), never "no data".

In [ ]:
q(r"""
SELECT table_schema, table_name
FROM system.information_schema.tables
WHERE table_catalog = 'system' AND table_schema <> 'information_schema'
ORDER BY table_schema, table_name
""")